In [1]:
import torch
from transformers import AutoTokenizer

import hparam as hp
from model import EmotionClassifierHybrid


In [ ]:
def load_classifier(checkpoint: Path, device: torch.device):
    tokenizer = AutoTokenizer.from_pretrained(hp.MODEL)
    model = EmotionClassifierHybrid(
        n_emotions=hp.NUM_EMOTIONS,
        input_dim=hp.input_dim,
        dropout=0.25,
        minilm=hp.MODEL,
        n_keep=0,
    )
    state_dict = torch.load(checkpoint, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return tokenizer, model

@torch.inference_mode()
def predict(texts: list[str], tokenizer, model, device: torch.device, top_k: int = 5):
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=hp.MAX_LEN,
        return_tensors="pt",
    ).to(device)

    logits = model(encoded["input_ids"], encoded["attention_mask"])
    probabilities = torch.sigmoid(logits).cpu()
    k = min(max(top_k, 1), len(hp.EMOTIONS))

    results = []
    for row in probabilities:
        values, indices = torch.topk(row, k=k)
        results.append(
            [(hp.EMOTIONS[index], float(value)) for value, index in zip(values, indices)]
        )
    return results


In [ ]:
tok, model = load_classifier("emotion_head_best_1.pt")

In [2]:
texts = ["I am so happy!", "I am so sad!!", "I am sad but also happy", "I am so horny!"]